# create models from ASVset genome objects

In [1]:
from load_genomes_to_modelseedpy import GenomeLoader
from cobra.io import write_sbml_model
# from glob import glob
from tqdm import tqdm
import os

loader = GenomeLoader()
# genome_ids = loader.list_available_genomes()#[:10]

def write_model(gID):
    model = loader.build_model(gID, printing=False)
    write_sbml_model(model, f'models/{gID}.xml')

# for genome_id in tqdm(genome_ids):
#     model_path = f'models/{genome_id}.xml'
#     if os.path.exists(model_path):  continue
#     try:
#         model = loader.build_model(genome_id, printing=False)
#         write_sbml_model(model, model_path)
#         # print(f"✓ {genome_id}") 
#     except Exception as e:
#         print(f"✗ {genome_id}: {e}")


# parallelize the effort
args = [gID for gID in loader.list_available_genomes() if not os.path.exists(f'models/{gID}.xml')]#[:2]
print(len(args), "will be processed")
parallelize = False
if parallelize:
    from multiprocess import Pool
    from os import cpu_count
    
    cpus = int(cpu_count()/8)
    print(f"{cpus} cores are being used.  The first argument is {args[0]}")
    pool = Pool(cpus)
    outputs = pool.map(write_model, args)
    
    print(f"missing iterativeIDs", missing_iterativeIDs)
    print(f"missing IDs", missing_IDs)
else:
    for arg in args:
        # print(arg)
        write_model(arg)

100%|██████████| 1698/1698 [1:21:00<00:00,  2.86s/it]  


# create community models from the member models and abundances

In [2]:
from cobra.io import read_sbml_model, write_sbml_model
from pandas import read_csv, Series
from mscommunity import MSCommunity
from tqdm import tqdm
from json import load
import os

# ab_df = read_csv("model_inputs/abundances.csv").set_index("seq")
# for sample in ab_df.columns:
#     print(sample, end=" ")
#     sample_abundances = ab_df[ab_df[sample] > 0]
#     print(len(sample_abundances))
iterativeIDs = load(open("modeling_files/iterativeIDs.json", 'r'))
ASV_sets = load(open("modeling_files/ASV_sets.json", 'r'))
redunant_asvs = {v:k for k,vs in ASV_sets.items() for v in vs}
abundances = load(open("modeling_files/ASVset_abundances.json", 'r'))
# missing_members = []
# for sample, ASVabun in tqdm(abundances.items()):
#     if os.path.exists(f"models/{sample}_comm.xml"):
#         continue
#     models = []
#     for ASV, abun in ASVabun.items():
#         ASVset = iterativeIDs[redunant_asvs.get(ASV, ASV)]
#         model_path = f"models/{ASVset}.xml"
#         #TODO:  investigate why these ASVs did not have any genomeIDs
#         if not os.path.exists(model_path):  # some ASVs have no genomeIDs
#             continue
            
#         # try:
#         model = read_sbml_model(model_path)
#         # except:
#         models.append(model)
#     comm = MSCommunity(member_models=models)
#     write_sbml_model(comm.util.model, f"models/{sample}_comm.xml")


def create_comm(item):
    sample = item[0]
    models = []
    for ASV, abun in item[1].items():
        ASVset = iterativeIDs[redunant_asvs.get(ASV, ASV)]
        model_path = f"models/{ASVset}.xml"
        #TODO:  investigate why these ASVs did not have any genomeIDs
        if not os.path.exists(model_path):  # some ASVs have no genomeIDs
            continue
            
        # try:
        model = read_sbml_model(model_path)
        # except:
        models.append(model)
    comm = MSCommunity(member_models=models)
    write_sbml_model(comm.util.model, f"models/{sample}_comm.xml")

# parallelize the effort
args = list([item for item in abundances.items() if not os.path.exists(f"models/{item[0]}_comm.xml")])
print(len(args), "will be processed")
parallelize = True
if parallelize:
    from multiprocess import Pool
    from os import cpu_count
    
    cpus = min(len(args), int(cpu_count()/8))
    print(f"{cpus} cores are being used.  The first argument is {args[0]}")
    pool = Pool(cpus)
    outputs = pool.map(create_comm, args)
    
    print(f"missing iterativeIDs", missing_iterativeIDs)
    print(f"missing IDs", missing_IDs)
else:
    for arg in args:
        # print(arg)
        create_comm(arg)

16 will be processed
16 cores are being used.  The first argument is ('C3', {'7f78afb7e1d45cd528fe9cd909c93fb5': 0.002491889872193666, '0fec351445f84bd482075ceda6123f46': 0.2580847663227067, '38b371de3be6ddef03bea94294e9830d': 0.03002708160249733, '63f37ee4b481fccdffca10160ed5b944': 0.03608264194234495, '35dbde30af820a2c3bcd85216229ad6a': 0.017615479571591992, 'd3cf08daefed96f494257ceff63a6cf8': 0.005776131827066854, 'a5b2537343a26f4daf0379a8b46eb2ef': 0.014099656454003483, '04869be25e3dc4d526ce3824d407aedf': 0.00887089828608216, '3343d19c0bc1759a66023cbc86c92592': 0.002951224418820203, 'c55a8101fb6c187736f0af61f720cb92': 0.10712638401711012, 'ec92f6331d8ecb50e3f0dee8fbf3bcb0': 0.030450052155473202, 'ea156f972cd3802c3fac147e4200ba7a': 0.0017875769120343642, '4bea53b2aad77f2569a0182f6b84255e': 0.0115312108257056, '83adaf3a09d08c93f2abea6a4c815592': 0.03351993798725384, '0709277372b3816e3986842fb10c8e96': 0.006700542593526704, 'e6485d233ed2b8b9a2f3f50f0747b9d2': 0.0039866410206298225, 'b

NameError: name 'missing_iterativeIDs' is not defined

In [4]:
from json import load
iterativeIDs = load(open("modeling_files/iterativeIDs.json", 'r'))
iterativeIDs

{'0fec351445f84bd482075ceda6123f46': 'Methanobacterium.1',
 '7f78afb7e1d45cd528fe9cd909c93fb5': 'Methanobacterium.2',
 '63f37ee4b481fccdffca10160ed5b944': 'Methanobacteriaceae.1',
 'd3cf08daefed96f494257ceff63a6cf8': 'Mesotoga.1',
 'c55a8101fb6c187736f0af61f720cb92': 'Lentimicrobium.1',
 '04869be25e3dc4d526ce3824d407aedf': 'Aminivibrio.1',
 '37a9f9625f201b6e61c29ac09ec54683': 'midas_g_94288.1',
 'b3d8bf326297fb9b18ea4fef1ce8d4a6': 'Desulfovibrio.1',
 '09ee5eb3e6c726c7c8242f6e97453fc2': 'Methanobacterium.3',
 '83adaf3a09d08c93f2abea6a4c815592': 'Petrimonas.1',
 'b64cf0e9dea8cef6b9500b364c179b98': 'Burkholderiales.1',
 '7281c911136ba2bdcafebb9c71729dea': 'Methanobacterium.4',
 '93fadef301b62aa732e42bf1142b3afa': 'midas_g_9269.1',
 '19463e5fe824700110e4a4ec8afe0512': 'Methanobacterium.5',
 'e6485d233ed2b8b9a2f3f50f0747b9d2': 'Proteiniphilum.1',
 'a5b2537343a26f4daf0379a8b46eb2ef': 'Syner-01.1',
 'f876e4d5cbaa6ac5f3c225bcaf146791': 'midas_g_2178.1',
 'a402fc6f0fd057decbe746f48bf12862': 'Pe

# Conduct community modeling

## Constraints

In [ ]:
from json import load, dump
from pandas import read_csv

